In [ ]:
import sys
import os

from joblib import load

# performance imports for torch: torch kernel uses one core only.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '../../../../../..')

# from reimplemented_approaches.proactive_conformance_checking.data_prep_split_encode import PrefixDataset
from reimplemented_approaches.proactive_conformance_checking.data_prep_split_encode_new import PrefixDataset
from reimplemented_approaches.proactive_conformance_checking.lstm_models import LSTMCollectiveIDP
from reimplemented_approaches.proactive_conformance_checking.evaluation import PredictionResults, Metrics

In [ ]:
data_dir = "../../data_preparation/Helpdesk/collective/"
path_model = "../../training/Helpdesk/collective/LSTM_collecctive_IDP.pkl"

In [ ]:
_, _, test_set = PrefixDataset.load_datasets(save_path=data_dir)

encoders = load(data_dir+"/encoders.pkl")

deviations = encoders.get("deviations")
print(deviations)

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device('cpu')
print(f"Evaluating on {device}")

In [ ]:
model = LSTMCollectiveIDP.load(path_model, device=device)

In [ ]:
pr = PredictionResults(model=model, test_set=test_set)
probs, preds, targets = pr.get_predictions_targets()
# print(preds)
# print(targets)
# print(probs)

In [ ]:
m = Metrics(preds=preds, targets=targets)

In [ ]:
res_dev = m.macro_precision_recall_dev()
print(res_dev)

In [ ]:
res_no_dev = m.macro_precision_recall_no_dev()
print(res_no_dev)

In [ ]:
pr_auc = m.plot_macro_pr_auc(prob_scores=probs, label_names=deviations)
print(pr_auc)